# NumPy and Matplotlib

Climate data is, underneath it all, **numbers in arrays** — a grid of model temperatures, a satellite swath, a stack of float profiles. **NumPy** is the library that lets Python store and compute on those arrays efficiently, and **Matplotlib** is how we turn them into figures. Almost everything later in this course (pandas, xarray) is built on top of these two, so this is the bedrock of the scientific Python stack.

In this notebook we'll create and index arrays, do math across whole arrays at once (vectorization and **broadcasting**), reduce them to summary statistics, and make our first plots.

- **NumPy** — *the fundamental package for scientific computing with Python* ([numpy.org](https://numpy.org/))
- **Matplotlib** — *a comprehensive library for static, animated, and interactive visualizations* ([matplotlib.org](https://matplotlib.org/))

:::{admonition} Working through this notebook
:class: tip
This page is a Jupyter notebook. **Download it** using the ⬇ button in the top-right of the page (or copy-paste the cells into a fresh notebook), open it in your environment (JupyterLab on LEAP or Colab), and step through the cells. When you reach a **Try it** admonition, experiment in your own cells before moving on.
:::

:::{admonition} In-class assignment — 10 points
:class: note
The **"Try it"** exercises in this notebook are part of your in-class assignment for this section. Complete them in your own copy of the notebook, push it to your week folder, and post the notebook link on the matching **Courseworks** assignment. (One 10-point assignment covers all the lecture notebooks in this section.)
:::

## Importing and Examining a New Package

This will be our first experience with _importing_ a package that is not part of the Python [standard library](https://docs.python.org/3/library/).

In [ ]:
import numpy as np

What did we just do? We **imported** a package. This makes new variables (mostly functions) available to use in our notebook — we can access them with the `np.` prefix, like this:

We can use `dir()` to see the variables we currently have access to — `np` should now be one of them.

In [ ]:
dir()

NumPy has hundreds of functions and types — `dir(np)` lists them all (too many to print here). A common pattern is to filter that list for names containing a keyword you care about:

In [ ]:
[name for name in dir(np) if 'lin' in name]

You can check the version of any package via its `__version__` attribute — useful when reproducing results or filing bug reports:

In [ ]:
np.__version__

There is no way we could explicitly teach each of these functions. The numpy documentation is crucial!

<https://numpy.org/doc/stable/reference/>

:::{admonition} Try it
:class: tip
Import the `math` module (`import math`). Use `dir(math)` to peek at what's inside. Pick a function that looks interesting (e.g., `log`, `factorial`, `sqrt`) and call `help()` on it to read its signature and docstring.
:::

## NDArrays

The core class is the numpy ndarray (n-dimensional array).

Compared to a regular Python `list`, numpy arrays have a few key differences:

- NumPy arrays can have N dimensions (lists, tuples, etc. are 1D).
- NumPy arrays hold values of a single datatype (e.g. `int`, `float`); `list`s can hold anything.
- NumPy optimizes numerical operations on arrays — NumPy is _fast!_

Let's create a 1D array from a Python list, then inspect its properties.

In [ ]:
a = np.array([9, 0, 2, 1, 0])
a

In [ ]:
a.dtype

In [ ]:
a.shape

The shape is returned as a **tuple**, which means we can index it like any other tuple — useful when we want to grab a specific dimension's length.

In [ ]:
type(a.shape)

Arrays can also be **multi-dimensional** and have an explicit dtype.

In [ ]:
b = np.array([[5, 3, 1, 9], [9, 2, 3, 0]], dtype=np.float64)
b.dtype, b.shape

:::{note}
The fastest-varying dimension is the **last** dimension; the outer level of the hierarchy is the first dimension. (This is called "C-style" indexing.)
:::

:::{admonition} Try it
:class: tip
Create three arrays:

- A 1D array from `[1, 2, 3, 4, 5]`.
- A 2D array from `[[1, 2], [3, 4]]`.
- A 3D array of your choice (a list of lists of lists).

Print the `shape` and `dtype` of each. Then create a 1D array that includes a decimal value (e.g. `[1, 2, 3.5]`) and confirm the dtype is now `float64`.
:::

## Array Creation

There are several ways to build arrays in numpy beyond passing a list to `np.array`. The functions below come up constantly:

- `np.zeros` / `np.ones` / `np.full` — blank arrays of a given shape.
- `np.arange` / `np.linspace` / `np.logspace` — evenly-spaced values along a range.
- `np.meshgrid` — combine 1D arrays into 2D coordinate grids.

In [ ]:
c = np.zeros((9, 9))
d = np.ones((3, 6, 3), dtype=np.complex128)
e = np.full((3, 3), np.pi)
e = np.ones_like(c)
f = np.zeros_like(d)

`arange` works very similar to `range`, but it populates the array "eagerly" (i.e. immediately), rather than generating the values upon iteration.

In [ ]:
np.arange(10)

`arange` is left inclusive, right exclusive, just like `range`, but also works with floating-point numbers.

In [ ]:
np.arange(2,4,0.25)

A frequent need is to generate an array of N numbers, evenly spaced between two values. That is what `linspace` is for.

In [ ]:
np.linspace(2,4,20)

Log-spaced values are useful when working with quantities spanning multiple orders of magnitude (e.g., frequencies, energy scales).

In [ ]:
np.logspace(1, 2, 10)

NumPy also has some utilities for helping us generate multi-dimensional arrays.
`meshgrid` creates 2D arrays out of a combination of 1D arrays. Meshgrid essentially tiles 1D arrays to a shape that is combined shape of the input arrays. 

In [ ]:
x = np.linspace(-2*np.pi, 2*np.pi, 100)
y = np.linspace(-np.pi, np.pi, 50)
xx, yy = np.meshgrid(x, y)
print(x.shape, y.shape)
print(xx.shape, yy.shape)

:::{admonition} Try it
:class: tip
Use `np.linspace` to create an array of 100 evenly-spaced values between 0 and 1, and `np.arange` to create the integers from 0 to 10. Print the `dtype` and `shape` of each. Then use `np.meshgrid` to build a 2D grid from these two arrays and print its shape.
:::

## Indexing

Basic indexing in numpy is similar to lists — single brackets, zero-based, and negative indices count from the end. The key extension is that for N-dimensional arrays, you separate the index for each dimension with a comma. Slicing (e.g. `[2:5]`, `[:, 0]`) and boolean masks also work, as we'll see below.

In [ ]:
y[10], y[1:5], y[-5]

For multi-dimensional arrays, use a comma to separate the index for each dimension.

In [ ]:
xx[0, 0], xx[-1, -1], xx[3, -5]

Slicing returns whole rows or columns.

In [ ]:
xx[0].shape, xx[:, -1].shape

You can take **ranges** across multiple dimensions.

In [ ]:
xx[3:10, 30:40].shape

There are many advanced ways to index arrays. You can [read about them](https://numpy.org/doc/stable/reference/arrays.indexing.html) in the manual. Here is one example.

In [ ]:
idx = xx < 0
xx[idx].shape

Note that a boolean index always returns a **flat (1D) array**, regardless of the input shape. The `ravel` method does the same flattening explicitly.

In [ ]:
xx.ravel().shape

:::{admonition} Try it
:class: tip
Take `xx` from the cells above. Get its first row, its last column, and a central 3×3 block via slicing. Then use a **boolean mask** to extract all values of `xx` that are greater than zero — what's the shape of the result?
:::

## Visualizing Arrays with Matplotlib

It can be hard to work with big arrays without actually seeing anything with our eyes!
We will now bring in Matplotlib to start visualizing these arrays.

In [ ]:
from matplotlib import pyplot as plt

For plotting a 1D array as a line, we use the `plot` command.

In [ ]:
plt.plot(x)

There are many ways to visualize 2D data.
He we use `pcolormesh`.

In [ ]:
plt.pcolormesh(xx)

In [ ]:
plt.pcolormesh(yy)

:::{admonition} Try it
:class: tip
Use `plt.plot` to plot `np.cos(x)` against `x`. Then use `plt.pcolormesh` to visualize the 2D array `xx + yy`. (Don't worry about labels yet — those come in the next notebook.)
:::

## Array Operations ##

There are a huge number of operations available on arrays. All the familiar arithmetic operators are applied on an element-by-element basis.

### Basic Math

In [ ]:
f = np.sin(x)

In [ ]:
plt.plot(x, f)

Now let's compute a function of two variables — a 2D surface. We use the 2D `xx`/`yy` arrays (from `meshgrid` earlier) so the function is evaluated at every grid point.

In [ ]:
f = np.sin(xx) * np.cos(0.5 * yy)

In [ ]:
plt.pcolormesh(xx,yy,f)

:::{admonition} Try it
:class: tip
Compute a 2D Gaussian on the `xx`/`yy` grid: `gaussian = np.exp(-(xx**2 + yy**2))`. Visualize it with `plt.pcolormesh`. Where is the array largest, and where is it smallest?
:::

## Manipulating array dimensions

Once you have an array, you often need to reshape it without changing the underlying data — to swap dimension order (transpose), reorganize the layout (`reshape`), repeat the array (`tile`), or add a new axis (with `None`) so it lines up with a higher-dimensional array. These all return a *view* or new array; they don't usually copy the data.

Swapping the dimension order is accomplished by calling `transpose`.

In [ ]:
f_transposed = f.transpose()
plt.pcolormesh(f_transposed)

We can also manually change the shape of an array...as long as the new shape has the same number of elements.

In [ ]:
g = np.reshape(f, (8,9))

However, be careful with reshapeing data!
You can accidentally lose the structure of the data.

In [ ]:
g = np.reshape(f, (200,25))
plt.pcolormesh(g)

We can also "tile" an array to repeat it many times.

In [ ]:
f_tiled = np.tile(f,(3, 2))
plt.pcolormesh(f_tiled)

Another common need is to add an extra dimension to an array.
This can be accomplished via indexing with `None`.

In [ ]:
x.shape

In [ ]:
x[None, :].shape

In [ ]:
x[None, :, None, None].shape

:::{admonition} Try it
:class: tip
Take the 2D array `f`. Transpose it and print the shape. Then reshape it to `(10, 500)` and print the shape. Finally, add a new leading axis so the shape becomes `(1, 50, 100)` — verify with `.shape`.
:::

## Broadcasting


Not all the arrays we want to work with will have the same size.
One approach would be to manually "expand" our arrays to all be the same size, e.g. using `tile`.
_Broadcasting_ is a more efficient way to multiply arrays of different sizes
NumPy has specific rules for how broadcasting works.
These can be confusing but are worth learning if you plan to work with NumPy data a lot.

The core concept of broadcasting is telling NumPy which dimensions are supposed to line up with each other.

![NumPy broadcasting rules](https://scipy-lectures.org/_images/numpy_broadcasting.png)

General Broadcasting Rules:
When performing operations on two arrays, NumPy compares their shapes element-wise starting from the trailing dimensions (right to left). It follows these rules:

- If the dimensions are equal, they're compatible.

- If one of the dimensions is 1, it's "stretched" to match the other.

- If the dimensions are unequal and neither is 1, the arrays are not broadcastable.

In [ ]:
print(f.shape, x.shape)
g = f * x
print(g.shape)

In [ ]:
plt.pcolormesh(f)

In [ ]:
plt.pcolormesh(g)

However, if the last two dimensions are _not_ the same, NumPy cannot just automatically figure it out.

In [ ]:
print(f.shape, y.shape)
h = f * y

We can help numpy by adding an extra dimension to `y` at the end.
Then the length-50 dimensions will line up.

In [ ]:
print(f.shape, y[:, None].shape)
h = f * y[:, None]
print(h.shape)

In [ ]:
plt.pcolormesh(h)

:::{admonition} Try it
:class: tip
Create and plot `f = np.sin(xx) * np.cos(0.5*yy)` from before — but use the **1D** `x` and `y` arrays plus broadcasting (add a `None` axis on each in the right place). Verify your result looks the same as the 2D version above.
:::

## Reduction Operations

In scientific data analysis, we usually start with a lot of data and want to reduce it down in order to make plots of summary tables.
Operations that reduce the size of numpy arrays are called "reductions".
There are many different reduction operations. Here we will look at some of the most common ones.

The usual statistical reductions — sum, mean, standard deviation — work as you'd expect when applied to the whole array:

In [ ]:
g.sum()

In [ ]:
g.mean()

In [ ]:
g.std()

A key property of numpy reductions is the ability to operate on just one axis.

In [ ]:
plt.pcolormesh(g)
plt.colorbar()

We can reduce along **specific axes**. `axis=0` reduces along the first dimension (rows of `g`), leaving the column means. `axis=1` reduces along the second dimension, leaving the row means.

In [ ]:
g_ymean = g.mean(axis=0)
g_xmean = g.mean(axis=1)

In [ ]:
plt.plot(x, g_ymean)

In [ ]:
plt.plot(g_xmean, y)

Reductions can also operate on **multiple axes at once** — useful for higher-dimensional data.

In [ ]:
arr3d = np.ones((100, 50, 25))
arr3d.mean(axis=(0, 1, 2))  # reduce across all three axes

:::{admonition} Try it
:class: tip
Take `g`. Compute its overall sum, mean, and standard deviation. Then take the mean **along axis 0** (the column means) and plot it against `x`. Take the standard deviation **along axis 1** (the row stds) and plot it against `y`.
:::

## Data Files

It's often useful to save a numpy array to disk so you can reload it later or share it with someone else. NumPy provides a simple `.npy` binary format for this, plus loaders for plain text and other common formats.

In [ ]:
np.save('g.npy', g)

:::{warning}
NumPy `.npy` files are convenient for temporary data, but are not a robust archival format. Later we'll meet NetCDF, the recommended format for earth and environmental data.
:::

In [ ]:
g_loaded = np.load('g.npy')

In [ ]:
g_loaded

To confirm that `assert_equal` really checks element-wise equality, here's a **deliberate failure** — we scale `g_loaded` by `0.5` first, so the arrays no longer match:

In [ ]:
np.testing.assert_equal(g, g_loaded*0.5)

In [ ]:
np.testing.assert_equal(g, g_loaded)

No output means the assertion **passed** — the loaded array is element-wise equal to the original.

:::{admonition} Try it
:class: tip
Save the 3D array `arr3d` to a file called `arr3d.npy`. Load it back into a new variable and verify they're equal using `np.testing.assert_equal`.
:::

## Recap

You've now met **NumPy**, the foundation of numerical Python:

- **Arrays (`ndarray`)** — creating them (`np.array`, `zeros`, `ones`, `arange`, `linspace`) and inspecting `shape` and `dtype`.
- **Indexing and slicing** — pulling out elements, rows, columns, and sub-arrays.
- **Vectorized operations** — math across whole arrays at once, including **broadcasting** between different shapes.
- **Reductions** — collapsing arrays with `sum`, `mean`, `max`, … along chosen axes.
- **Plotting and files** — a first look at Matplotlib, and saving/loading arrays with `np.save`/`np.load`.

Next we go deeper into visualization — the Figure/Axes model and the full range of plot types — in [More Matplotlib](./more_matplotlib.ipynb).